# Phase 5 — Paper-Ready Validation
**Weather-aware multimodal crop disease classification | Google Colab**

This notebook is designed to run after Phase 4. It retrains the selected model configurations over multiple random seeds, reports mean ± standard deviation, produces class-wise metrics and confusion matrices, runs paired test-set comparisons, and generates image/weather interpretability outputs.

**Default protocol**
- 3 seeds by default (set `SEEDS = [42, 123, 2025, 7, 99]` for five runs).
- Uses the same Phase 2 split CSVs and the same weather-window convention as Phase 4.
- Selects checkpoints using validation macro-F1; test data is used only for final evaluation.
- Reports measured results only. No placeholder scores are generated.

**Scientific caution:** repeated seeds are not independent datasets. The paired test-set analysis complements, but does not replace, repeated-seed reporting and careful leakage checks.

## 1. Mount Drive and configure paths
Run in Colab with a GPU runtime. This notebook is self-contained; it redefines the model and training utilities instead of relying on variables from a previous notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, time, random, copy, warnings
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, f1_score
)
from scipy.stats import binomtest, ttest_rel

PROJECT_ROOT = Path("/content/drive/MyDrive")
DATASET_ROOT = PROJECT_ROOT / "DATASET"
PHASE2_DIR = DATASET_ROOT / "docs" / "phase2_outputs"
PHASE4_DIR = DATASET_ROOT / "docs" / "phase4_outputs"
OUTPUT_DIR = DATASET_ROOT / "docs" / "phase5_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Phase 2 inputs:", PHASE2_DIR)
print("Phase 4 results:", PHASE4_DIR)
print("Phase 5 outputs:", OUTPUT_DIR)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Experiment configuration

In [ ]:
# Run at least 3 seeds; use 5 for stronger evidence if compute permits.
SEEDS = [42, 123, 2025]  # For five seeds: [42, 123, 2025, 7, 99]

IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
EPOCHS = 10
PATIENCE = 3
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
PRETRAINED = True
FREEZE_BACKBONE = True
BACKBONE_NAME = "convnext_tiny"  # or "efficientnet_b0"
WEATHER_WINDOW_DAYS = 7
WEATHER_COLUMNS = {
    1: ["temp_mean_1d", "humidity_mean_1d", "rainfall_sum_1d"],
    3: ["temp_mean_3d", "humidity_mean_3d", "rainfall_sum_3d"],
    7: ["temp_mean_7d", "humidity_mean_7d", "rainfall_sum_7d"],
    14: ["temp_mean_14d", "humidity_mean_14d", "rainfall_sum_14d"],
}[WEATHER_WINDOW_DAYS]

# Choose candidates based on Phase 4 validation results. Keep image_only as control.
MODELS_TO_VALIDATE = ["image_only", "concat", "late_fusion", "gated_fusion", "film"]
USE_CLASS_WEIGHTS = True
MIXED_PRECISION = True

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = MIXED_PRECISION and device.type == "cuda"
print("Device:", device, "| Window:", WEATHER_WINDOW_DAYS, "days | Weather:", WEATHER_COLUMNS)

## 3. Load and audit the Phase 2 manifests

In [ ]:
train_df = pd.read_csv(PHASE2_DIR / "multimodal_train.csv")
val_df = pd.read_csv(PHASE2_DIR / "multimodal_val.csv")
test_df = pd.read_csv(PHASE2_DIR / "multimodal_test.csv")

required = {"image_path", "label", "split_group", *WEATHER_COLUMNS}
for split_name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{split_name} CSV is missing columns: {sorted(missing)}")
    if frame.empty:
        raise ValueError(f"{split_name} CSV is empty.")
    if frame[WEATHER_COLUMNS].isna().any().any():
        raise ValueError(f"Missing weather values in {split_name}; clean/impute Phase 2 data first.")

all_df = pd.concat([
    train_df.assign(_split="train"),
    val_df.assign(_split="val"),
    test_df.assign(_split="test")
], ignore_index=True)
group_splits = all_df.groupby("split_group")["_split"].nunique()
if (group_splits > 1).any():
    raise ValueError("Leakage check failed: split_group occurs across train/val/test.")

class_names = sorted(train_df["label"].astype(str).unique().tolist())
label_to_idx = {c: i for i, c in enumerate(class_names)}
for frame in (train_df, val_df, test_df):
    frame["label"] = frame["label"].astype(str)
    unknown = set(frame["label"]) - set(class_names)
    if unknown:
        raise ValueError(f"Labels absent from training split: {unknown}")
    frame["label_idx"] = frame["label"].map(label_to_idx).astype(int)

print("Classes:", len(class_names))
print("Split sizes:", len(train_df), len(val_df), len(test_df))
display(pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts()
}).fillna(0).astype(int))

## 4. Normalize weather using training data only

In [ ]:
weather_mean = train_df[WEATHER_COLUMNS].astype(float).mean()
weather_std = train_df[WEATHER_COLUMNS].astype(float).std().replace(0, 1).fillna(1)
config = {
    "seeds": SEEDS, "backbone": BACKBONE_NAME,
    "weather_window_days": WEATHER_WINDOW_DAYS,
    "weather_columns": WEATHER_COLUMNS, "class_names": class_names,
    "weather_mean": weather_mean.to_dict(), "weather_std": weather_std.to_dict(),
    "models": MODELS_TO_VALIDATE, "epochs": EPOCHS
}
(OUTPUT_DIR / "phase5_config.json").write_text(json.dumps(config, indent=2))
display(pd.DataFrame({"mean": weather_mean, "std": weather_std}))

## 5. Dataset and data loaders

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08),
    transforms.ToTensor(), transforms.Normalize(imagenet_mean, imagenet_std)
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(), transforms.Normalize(imagenet_mean, imagenet_std)
])

def resolve_image_path(value):
    p = Path(str(value))
    candidates = [p, DATASET_ROOT / p, PROJECT_ROOT / p]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Image not found: {value}. Verify image_path values and DATASET_ROOT.")

class CropWeatherDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(resolve_image_path(row["image_path"])) as im:
            image = self.transform(im.convert("RGB"))
        weather = (row[WEATHER_COLUMNS].astype(float).values - weather_mean.values) / weather_std.values
        return image, torch.tensor(weather, dtype=torch.float32), torch.tensor(int(row["label_idx"]), dtype=torch.long)

train_ds = CropWeatherDataset(train_df, train_transform)
val_ds = CropWeatherDataset(val_df, eval_transform)
test_ds = CropWeatherDataset(test_df, eval_transform)

def make_loaders(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=generator,
                              num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                            pin_memory=torch.cuda.is_available())
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                             pin_memory=torch.cuda.is_available())
    return train_loader, val_loader, test_loader

xb, wb, yb = next(iter(make_loaders(SEEDS[0])[0]))
print("Batch shapes:", xb.shape, wb.shape, yb.shape)

## 6. Define model architectures

In [ ]:
def build_image_encoder(name=BACKBONE_NAME, pretrained=PRETRAINED, freeze=FREEZE_BACKBONE):
    if name == "convnext_tiny":
        weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        net = models.convnext_tiny(weights=weights)
        feature_dim = net.classifier[2].in_features
        net.classifier = nn.Identity()
        encoder = net
    elif name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        net = models.efficientnet_b0(weights=weights)
        feature_dim = net.classifier[1].in_features
        net.classifier = nn.Identity()
        encoder = net
    else:
        raise ValueError("Unsupported backbone.")
    if freeze:
        for p in encoder.parameters():
            p.requires_grad = False
    return encoder, feature_dim

class WeatherEncoder(nn.Module):
    def __init__(self, input_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(64, hidden), nn.LayerNorm(hidden), nn.GELU()
        )
    def forward(self, x):
        return self.net(x)

class MultimodalClassifier(nn.Module):
    def __init__(self, mode, num_classes):
        super().__init__()
        self.mode = mode
        self.image_encoder, self.image_dim = build_image_encoder()
        self.weather_encoder = WeatherEncoder(len(WEATHER_COLUMNS), 128)
        self.image_head = nn.Sequential(nn.Dropout(0.25), nn.Linear(self.image_dim, num_classes))
        self.weather_head = nn.Linear(128, num_classes)
        self.fused_head = nn.Sequential(
            nn.LayerNorm(self.image_dim + 128), nn.Dropout(0.3),
            nn.Linear(self.image_dim + 128, 256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        self.gate = nn.Sequential(
            nn.Linear(self.image_dim + 128, 256), nn.GELU(),
            nn.Linear(256, 1), nn.Sigmoid()
        )
        self.weather_to_image = nn.Linear(128, self.image_dim)
        self.film = nn.Linear(128, self.image_dim * 2)
        self.film_head = nn.Sequential(
            nn.LayerNorm(self.image_dim), nn.Dropout(0.3),
            nn.Linear(self.image_dim, num_classes)
        )
        self.late_scale = nn.Parameter(torch.tensor(0.0))
    def forward(self, images, weather):
        image_features = self.image_encoder(images)
        if image_features.ndim > 2:
            image_features = torch.flatten(image_features, 1)
        image_logits = self.image_head(image_features)
        if self.mode == "image_only":
            return image_logits
        weather_features = self.weather_encoder(weather)
        weather_logits = self.weather_head(weather_features)
        if self.mode == "concat":
            return self.fused_head(torch.cat([image_features, weather_features], dim=1))
        if self.mode == "late_fusion":
            alpha = torch.sigmoid(self.late_scale)
            return alpha * image_logits + (1 - alpha) * weather_logits
        if self.mode == "gated_fusion":
            gate = self.gate(torch.cat([image_features, weather_features], dim=1))
            fused = gate * image_features + (1 - gate) * self.weather_to_image(weather_features)
            return self.film_head(fused)
        if self.mode == "film":
            gamma, beta = self.film(weather_features).chunk(2, dim=1)
            return self.film_head(image_features * (1 + torch.tanh(gamma)) + beta)
        raise ValueError(f"Unknown mode: {self.mode}")

## 7. Train with validation-based checkpoint selection

In [ ]:
def get_class_weights():
    counts = train_df["label_idx"].value_counts().reindex(range(len(class_names)), fill_value=0).values.astype(np.float32)
    counts = np.maximum(counts, 1)
    return torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float32, device=device)

def run_epoch(model, loader, optimizer=None, criterion=None):
    training = optimizer is not None
    model.train(training)
    losses, ys, ps = [], [], []
    for images, weather, targets in loader:
        images, weather, targets = images.to(device), weather.to(device), targets.to(device)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
                logits = model(images, weather)
                loss = criterion(logits, targets) if criterion is not None else F.cross_entropy(logits, targets)
            if training:
                loss.backward()
                optimizer.step()
        losses.append(float(loss.detach()) * len(targets))
        ys.extend(targets.detach().cpu().tolist())
        ps.extend(logits.argmax(1).detach().cpu().tolist())
    p, r, f1, _ = precision_recall_fscore_support(ys, ps, average="macro", zero_division=0)
    return {"loss": sum(losses)/len(loader.dataset), "accuracy": accuracy_score(ys, ps),
            "precision_macro": p, "recall_macro": r, "macro_f1": f1,
            "targets": ys, "predictions": ps}

def train_one(mode, seed):
    seed_everything(seed)
    train_loader, val_loader, test_loader = make_loaders(seed)
    model = MultimodalClassifier(mode, len(class_names)).to(device)
    criterion = nn.CrossEntropyLoss(weight=get_class_weights() if USE_CLASS_WEIGHTS else None)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                  lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    best_f1, best_state, bad_epochs = -1, None, 0
    history = []
    ckpt = OUTPUT_DIR / f"best_{mode}_{BACKBONE_NAME}_{WEATHER_WINDOW_DAYS}d_seed{seed}.pt"
    for epoch in range(1, EPOCHS + 1):
        tr = run_epoch(model, train_loader, optimizer, criterion)
        va = run_epoch(model, val_loader, criterion=criterion)
        history.append({
            "model": mode, "seed": seed, "epoch": epoch,
            "train_loss": tr["loss"], "train_accuracy": tr["accuracy"], "train_macro_f1": tr["macro_f1"],
            "val_loss": va["loss"], "val_accuracy": va["accuracy"], "val_macro_f1": va["macro_f1"]
        })
        print(f"{mode} seed={seed} epoch={epoch}: val acc={va['accuracy']:.4f}, macro-F1={va['macro_f1']:.4f}")
        if va["macro_f1"] > best_f1:
            best_f1, bad_epochs = va["macro_f1"], 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save({
                "model_state_dict": best_state, "mode": mode, "seed": seed,
                "class_names": class_names, "weather_columns": WEATHER_COLUMNS,
                "weather_mean": weather_mean.to_dict(), "weather_std": weather_std.to_dict(),
                "backbone": BACKBONE_NAME, "image_size": IMAGE_SIZE,
                "weather_window_days": WEATHER_WINDOW_DAYS
            }, ckpt)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break
    model.load_state_dict(best_state)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / f"history_{mode}_{WEATHER_WINDOW_DAYS}d_seed{seed}.csv", index=False)
    val_metrics = run_epoch(model, val_loader, criterion=criterion)
    test_metrics = run_epoch(model, test_loader, criterion=criterion)
    return model, ckpt, history, val_metrics, test_metrics

## 8. Run multi-seed validation

In [ ]:
all_results = []
histories = {}
models_by_key = {}
predictions_by_key = {}
targets_by_key = {}

for mode in MODELS_TO_VALIDATE:
    for seed in SEEDS:
        print("\n" + "="*70, "\n", mode, "| seed", seed)
        model, ckpt, history, val_metrics, test_metrics = train_one(mode, seed)
        key = (mode, seed)
        models_by_key[key] = model
        histories[key] = history
        predictions_by_key[key] = test_metrics["predictions"]
        targets_by_key[key] = test_metrics["targets"]
        all_results.append({
            "model": mode, "seed": seed,
            "best_val_macro_f1": max(h["val_macro_f1"] for h in history),
            "val_accuracy": val_metrics["accuracy"], "val_macro_f1": val_metrics["macro_f1"],
            "test_accuracy": test_metrics["accuracy"],
            "test_precision_macro": test_metrics["precision_macro"],
            "test_recall_macro": test_metrics["recall_macro"],
            "test_macro_f1": test_metrics["macro_f1"],
            "checkpoint": str(ckpt)
        })
        print("Saved:", ckpt)

results_df = pd.DataFrame(all_results)
results_df.to_csv(OUTPUT_DIR / "phase5_seed_level_results.csv", index=False)
display(results_df)


 image_only | seed 42
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 182MB/s] 


image_only seed=42 epoch=1: val acc=0.7941, macro-F1=0.7250
image_only seed=42 epoch=2: val acc=0.7993, macro-F1=0.7066
image_only seed=42 epoch=3: val acc=0.8149, macro-F1=0.7350
image_only seed=42 epoch=4: val acc=0.8131, macro-F1=0.7338
image_only seed=42 epoch=5: val acc=0.8201, macro-F1=0.7486
image_only seed=42 epoch=6: val acc=0.8235, macro-F1=0.7644
image_only seed=42 epoch=7: val acc=0.8235, macro-F1=0.7679
image_only seed=42 epoch=8: val acc=0.8304, macro-F1=0.7814
image_only seed=42 epoch=9: val acc=0.8356, macro-F1=0.7897
image_only seed=42 epoch=10: val acc=0.8339, macro-F1=0.7881
Saved: /content/drive/MyDrive/DATASET/docs/phase5_outputs/best_image_only_convnext_tiny_7d_seed42.pt

 image_only | seed 123
image_only seed=123 epoch=1: val acc=0.7716, macro-F1=0.6835


## 9. Mean ± standard deviation across seeds

In [ ]:
metric_cols = ["test_accuracy", "test_precision_macro", "test_recall_macro", "test_macro_f1",
               "val_accuracy", "val_macro_f1"]
summary = results_df.groupby("model")[metric_cols].agg(["mean", "std"])
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.reset_index()
summary.to_csv(OUTPUT_DIR / "phase5_mean_std_summary.csv", index=False)

pretty = summary[["model"]].copy()
for metric in ["test_accuracy", "test_precision_macro", "test_recall_macro", "test_macro_f1"]:
    pretty[metric + " (mean ± std)"] = summary.apply(
        lambda r: f'{r[metric+"_mean"]:.4f} ± {0 if pd.isna(r[metric+"_std"]) else r[metric+"_std"]:.4f}', axis=1)
display(pretty)

## 10. Paired test-set statistical comparisons

The following tests compare predictions on the **same test examples**:
- McNemar's exact binomial test compares paired correctness/errors for two classifiers.
- Paired bootstrap estimates a 95% confidence interval for the difference in macro-F1 by resampling test examples.

These tests assess the specific saved model pair on this test set. They do not imply generalization to every farm, location, season or dataset.

In [ ]:
def paired_mcnemar(y_true, pred_a, pred_b):
    y_true = np.asarray(y_true); pred_a = np.asarray(pred_a); pred_b = np.asarray(pred_b)
    correct_a = pred_a == y_true
    correct_b = pred_b == y_true
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    n = b + c
    p_value = 1.0 if n == 0 else binomtest(min(b, c), n=n, p=0.5, alternative="two-sided").pvalue
    return {"a_correct_b_wrong": b, "a_wrong_b_correct": c, "mcnemar_exact_p": p_value}

def paired_bootstrap_macro_f1(y_true, pred_a, pred_b, n_boot=2000, seed=123):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true); pred_a = np.asarray(pred_a); pred_b = np.asarray(pred_b)
    diffs = []
    n = len(y_true)
    for _ in range(n_boot):
        ix = rng.integers(0, n, size=n)
        diffs.append(
            f1_score(y_true[ix], pred_b[ix], average="macro", zero_division=0) -
            f1_score(y_true[ix], pred_a[ix], average="macro", zero_division=0)
        )
    observed = f1_score(y_true, pred_b, average="macro", zero_division=0) - f1_score(y_true, pred_a, average="macro", zero_division=0)
    lo, hi = np.quantile(diffs, [0.025, 0.975])
    return observed, lo, hi

# Select the best multimodal architecture by mean validation macro-F1 (not test scores).
val_summary = results_df.groupby("model")["val_macro_f1"].mean().sort_values(ascending=False)
multimodal_names = [m for m in MODELS_TO_VALIDATE if m != "image_only"]
best_multimodal = max(multimodal_names, key=lambda m: val_summary.get(m, -1)) if multimodal_names else None
print("Validation-selected multimodal model:", best_multimodal)

# Compare one fixed seed's paired predictions; repeat with a prespecified seed if needed.
comparison_seed = SEEDS[0]
if best_multimodal:
    y = np.asarray(targets_by_key[("image_only", comparison_seed)])
    pred_img = np.asarray(predictions_by_key[("image_only", comparison_seed)])
    pred_multi = np.asarray(predictions_by_key[(best_multimodal, comparison_seed)])
    mcnemar = paired_mcnemar(y, pred_img, pred_multi)
    delta, ci_low, ci_high = paired_bootstrap_macro_f1(y, pred_img, pred_multi)
    paired_result = {
        "comparison_seed": comparison_seed, "baseline": "image_only", "multimodal": best_multimodal,
        **mcnemar, "macro_f1_delta_multimodal_minus_image": delta,
        "bootstrap_95ci_low": ci_low, "bootstrap_95ci_high": ci_high
    }
    pd.DataFrame([paired_result]).to_csv(OUTPUT_DIR / "phase5_paired_test_comparison.csv", index=False)
    display(pd.DataFrame([paired_result]))

## 11. Class-wise metrics and confusion matrices

In [ ]:
# Aggregate out-of-fold-style class reports across each model's seed predictions.
# Each seed's report is saved separately; the displayed aggregate is the mean class-wise metric over seeds.
class_rows = []
for mode in MODELS_TO_VALIDATE:
    reports = []
    for seed in SEEDS:
        y = np.asarray(targets_by_key[(mode, seed)])
        pred = np.asarray(predictions_by_key[(mode, seed)])
        rep = classification_report(y, pred, labels=list(range(len(class_names))),
                                    target_names=class_names, output_dict=True, zero_division=0)
        for cls in class_names:
            class_rows.append({
                "model": mode, "seed": seed, "class": cls,
                "precision": rep[cls]["precision"], "recall": rep[cls]["recall"],
                "f1_score": rep[cls]["f1-score"], "support": rep[cls]["support"]
            })
        cm = confusion_matrix(y, pred, labels=list(range(len(class_names))))
        pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(
            OUTPUT_DIR / f"confusion_matrix_{mode}_seed{seed}.csv"
        )
        pd.DataFrame(rep).T.to_csv(OUTPUT_DIR / f"classification_report_{mode}_seed{seed}.csv")

class_df = pd.DataFrame(class_rows)
class_df.to_csv(OUTPUT_DIR / "phase5_classwise_metrics_by_seed.csv", index=False)
class_summary = class_df.groupby(["model", "class"])[["precision", "recall", "f1_score"]].agg(["mean", "std"])
class_summary.columns = [f"{a}_{b}" for a,b in class_summary.columns]
class_summary.reset_index().to_csv(OUTPUT_DIR / "phase5_classwise_mean_std.csv", index=False)
display(class_summary.reset_index().head(30))

## 12. Plot mean ± standard deviation

In [ ]:
plot_df = results_df.groupby("model").agg(
    mean_f1=("test_macro_f1", "mean"), std_f1=("test_macro_f1", "std"),
    mean_acc=("test_accuracy", "mean"), std_acc=("test_accuracy", "std")
).reset_index()
plot_df["std_f1"] = plot_df["std_f1"].fillna(0)
plot_df["std_acc"] = plot_df["std_acc"].fillna(0)

plt.figure(figsize=(10, 5))
plt.bar(plot_df["model"], plot_df["mean_f1"], yerr=plot_df["std_f1"], capsize=4)
plt.ylabel("Test macro-F1"); plt.title("Phase 5: Test macro-F1 across seeds")
plt.xticks(rotation=25, ha="right"); plt.grid(axis="y", alpha=0.25); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "phase5_macro_f1_mean_std.png", dpi=180)
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(plot_df["model"], plot_df["mean_acc"], yerr=plot_df["std_acc"], capsize=4)
plt.ylabel("Test accuracy"); plt.title("Phase 5: Test accuracy across seeds")
plt.xticks(rotation=25, ha="right"); plt.grid(axis="y", alpha=0.25); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "phase5_accuracy_mean_std.png", dpi=180)
plt.show()

## 13. Image explainability with Grad-CAM

Grad-CAM highlights image regions that influence the chosen class score. This implementation uses the final convolutional layer of the image encoder. It is a qualitative explanation, not proof that the model reasons causally or correctly.

In [ ]:
def find_last_conv_layer(module):
    last = None
    for layer in module.modules():
        if isinstance(layer, nn.Conv2d):
            last = layer
    if last is None:
        raise RuntimeError("No Conv2d layer found for Grad-CAM.")
    return last

def gradcam_overlay(model, image_tensor, weather_tensor, class_idx=None):
    model.eval()
    target_layer = find_last_conv_layer(model.image_encoder)
    activations, gradients = {}, {}
    def forward_hook(module, inp, out):
        activations["value"] = out
        out.register_hook(lambda grad: gradients.__setitem__("value", grad))
    handle = target_layer.register_forward_hook(forward_hook)
    x = image_tensor.unsqueeze(0).to(device).requires_grad_(True)
    w = weather_tensor.unsqueeze(0).to(device)
    model.zero_grad(set_to_none=True)
    logits = model(x, w)
    if class_idx is None:
        class_idx = int(logits.argmax(1).item())
    logits[0, class_idx].backward()
    handle.remove()
    acts = activations["value"].detach()
    grads = gradients["value"].detach()
    weights = grads.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((weights * acts).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
    cam = cam[0, 0].cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    # Convert normalized tensor back to displayable RGB.
    img = image_tensor.detach().cpu().permute(1,2,0).numpy()
    img = img * np.array(imagenet_std) + np.array(imagenet_mean)
    img = np.clip(img, 0, 1)
    return img, cam, class_idx

# Explain the validation-selected multimodal model for the first seed.
explain_mode = best_multimodal if best_multimodal else "image_only"
explain_seed = SEEDS[0]
explain_model = models_by_key[(explain_mode, explain_seed)]
sample_idx = 0
img_tensor, weather_tensor, true_label = test_ds[sample_idx]
img, cam, pred_class = gradcam_overlay(explain_model, img_tensor, weather_tensor)
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(img); plt.title(f"Input | true: {class_names[int(true_label)]}"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(img); plt.imshow(cam, cmap="jet", alpha=0.4)
plt.title(f"Grad-CAM | predicted: {class_names[pred_class]}"); plt.axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"gradcam_{explain_mode}_seed{explain_seed}.png", dpi=180)
plt.show()

## 14. Weather feature importance by permutation

For each weather feature, this section permutes that feature across test examples and measures the change in accuracy and macro-F1. A larger performance drop suggests greater model reliance on that feature. Correlated weather variables can make individual importances unstable; report this as a diagnostic.

In [ ]:
@torch.no_grad()
def predict_with_weather_array(model, dataset, weather_array, batch_size=BATCH_SIZE):
    model.eval()
    ys, ps = [], []
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    offset = 0
    for images, _, targets in loader:
        n = len(targets)
        images = images.to(device)
        weather = torch.tensor(weather_array[offset:offset+n], dtype=torch.float32, device=device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            logits = model(images, weather)
        ys.extend(targets.numpy().tolist())
        ps.extend(logits.argmax(1).cpu().numpy().tolist())
        offset += n
    return np.asarray(ys), np.asarray(ps)

weather_importance_rows = []
if explain_mode != "image_only":
    raw_weather = test_df[WEATHER_COLUMNS].astype(float).values
    standardized_weather = (raw_weather - weather_mean.values) / weather_std.values
    base_y, base_pred = predict_with_weather_array(explain_model, test_ds, standardized_weather)
    base_acc = accuracy_score(base_y, base_pred)
    base_f1 = f1_score(base_y, base_pred, average="macro", zero_division=0)
    rng = np.random.default_rng(2025)
    for j, col in enumerate(WEATHER_COLUMNS):
        drops_acc, drops_f1 = [], []
        for repeat in range(10):
            permuted = standardized_weather.copy()
            permuted[:, j] = permuted[rng.permutation(len(permuted)), j]
            y, pred = predict_with_weather_array(explain_model, test_ds, permuted)
            drops_acc.append(base_acc - accuracy_score(y, pred))
            drops_f1.append(base_f1 - f1_score(y, pred, average="macro", zero_division=0))
        weather_importance_rows.append({
            "feature": col, "accuracy_drop_mean": float(np.mean(drops_acc)),
            "accuracy_drop_std": float(np.std(drops_acc, ddof=1)),
            "macro_f1_drop_mean": float(np.mean(drops_f1)),
            "macro_f1_drop_std": float(np.std(drops_f1, ddof=1))
        })
    importance_df = pd.DataFrame(weather_importance_rows).sort_values("macro_f1_drop_mean", ascending=False)
    importance_df.to_csv(OUTPUT_DIR / "phase5_weather_permutation_importance.csv", index=False)
    display(importance_df)
    plt.figure(figsize=(8,4))
    plt.bar(importance_df["feature"], importance_df["macro_f1_drop_mean"],
            yerr=importance_df["macro_f1_drop_std"], capsize=4)
    plt.ylabel("Macro-F1 drop after permutation")
    plt.title("Weather feature permutation importance")
    plt.xticks(rotation=20, ha="right"); plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "phase5_weather_permutation_importance.png", dpi=180)
    plt.show()
else:
    print("Selected explanation model is image-only; weather permutation importance is not applicable.")

## 15. Automatically create paper-ready result tables and limitations notes

In [ ]:
# Compact CSV table with mean ± SD strings for the main test metrics.
paper_table = pd.DataFrame({"Model": summary["model"]})
for metric, label in [
    ("test_accuracy", "Accuracy"),
    ("test_precision_macro", "Macro precision"),
    ("test_recall_macro", "Macro recall"),
    ("test_macro_f1", "Macro-F1")
]:
    paper_table[label] = summary.apply(
        lambda r: f'{r[metric+"_mean"]:.4f} ± {0 if pd.isna(r[metric+"_std"]) else r[metric+"_std"]:.4f}', axis=1
    )
paper_table.to_csv(OUTPUT_DIR / "paper_table_main_results.csv", index=False)
display(paper_table)

limitations = [
    "Weather windows are aggregated features; they are not a learned daily temporal sequence.",
    "Weather pairing validity depends on correct date/location metadata and source coverage.",
    "Repeated random seeds do not replace external validation on independent locations, seasons or farms.",
    "Grad-CAM is qualitative and does not establish causal reasoning.",
    "Permutation importance can be affected by correlation among weather variables.",
    "Use only measured results; do not report unrun experiments or placeholder values.",
    "State split strategy and verify that near-duplicate images or related plants do not cross splits."
]
(OUTPUT_DIR / "paper_limitations_checklist.txt").write_text("\n".join(f"- {x}" for x in limitations))
print("Saved Phase 5 outputs to:", OUTPUT_DIR)
print("Key files: phase5_mean_std_summary.csv, phase5_paired_test_comparison.csv,")
print("phase5_classwise_mean_std.csv, paper_table_main_results.csv, Grad-CAM and weather importance figures.")